# Learn to fly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/teaching/drone_ppo_learn_to_fly.ipynb)

This notebook demonstrates **policy optimization** with a reinforcement learning algorithm that learns a feedback law $u=\pi_\theta(x)$ that minimizes a performance metric $J$ for a planar drone.

You can change the cost, the initial-state distribution, and the training budget, then inspect the learned control law and the closed-loop flight.

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox for the dynamics, simulation, and animation, and [stable-baselines3](https://stable-baselines3.readthedocs.io) for the RL algorithm (PPO). For the library workflow see [`showcase_minilink`](../intro/showcase_minilink.ipynb); plants and closed-loop diagrams are in [`02_dynamics`](../intro/02_dynamics.ipynb) and [`05_simulation`](../intro/05_simulation.ipynb).


In [ ]:
# Local: minilink already installed. Colab: clone + path + RL dependencies.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q gymnasium stable-baselines3")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink.core.costs import CostFunction
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.aerial.drone import Drone2D
from minilink.interfaces.gymnasium import SB3Controller, Sys2Gym

# Dynamics

We load a minilink catalog class (`Drone2D`) that already defines the equations of motion of a planar drone with two body-vertical thrusters. The state and input are
$$x = [x,\; y,\; \theta,\; v_x,\; v_y,\; \omega],\qquad u = [T_1,\; T_2],$$
and the dynamics have the form $\dot x = f(x,u)$. We also set the **domain** (bounds on $x$ and $u$) used by the gym environment.

To help the RL algorithm, inputs are normalized to $[-1,1]$ instead of Newtons. With this scaling, $u = [0,\; 0]$ is hover (gravity compensation), $u = [1,\; 1]$ is maximum thrust, and $u = [-1,\; -1]$ is minimum thrust.

In [ ]:
class NormalizedDrone2D(Drone2D):
    """Planar drone with thrust inputs normalized between -1 and 1."""

    def __init__(self):
        super().__init__()

        # Parameters
        self.params["mass"] = 1.0  # kg
        self.params["inertia"] = 0.1  # kgm2

        # Normalized inputs
        self.inputs["u"].lower_bound = np.array([-1.0, -1.0])
        self.inputs["u"].upper_bound = np.array([+1.0, +1.0])
        self.inputs["u"].units = ["%", "%"]

        self.weight = self.params["gravity"] * self.params["mass"]
        self.thrust2weight = 1.2

        # Min/max states
        self.state.upper_bound = np.array([10, 10, 2 * np.pi, 10, 10, 10])
        self.state.lower_bound = -self.state.upper_bound

    def thrust(self, u):
        """Map a normalized input to thruster forces in Newtons."""
        return self.weight * ((self.thrust2weight - 1.0) * u + np.array([0.5, 0.5]))

    def f(self, x, u, t=0.0, params=None):
        return super().f(x, self.thrust(u), t, params)

    def get_dynamic_geometry(self, x, u, t=0, params=None):
        # Draw the thrust arrows using the de-normalized forces
        return super().get_dynamic_geometry(x, self.thrust(u), t, params)


plant = NormalizedDrone2D()

Open-loop check of the plant: a small constant differential thrust, no feedback. This is only to see the EoM and the animation before we train a policy.

In [ ]:
plant.x0 = np.array([-5.0, -2.0, 0.1, 0.0, 0.0, 0.1])

traj_open = plant.compute_forced(
    u=lambda t: np.array([0.01, -0.01]), tf=5.0, n_steps=2001, solver="euler"
)
plant.plot_trajectory(traj_open)
plant.animate(traj_open)

# Cost function

The performance metric is a Bolza cost
$$J = \int_{0}^{t_f} g(x, u, t) \, dt + h(x_f, t_f).$$
The class below implements a quadratic running cost about the hover target $\bar x = 0$,
$$g(x,u) = (x-\bar x)' Q (x-\bar x) + u' R u,$$
with $h=0$. The RL algorithm maximizes the return $R = \sum r_t$ with $r = -g(x,u,t)\,\Delta t$, so maximizing $R$ is the same as minimizing $J$. Edit $Q$, $R$, or $g$ here to change the objective.

In [ ]:
class CustomCostFunction(CostFunction):
    """
    J = int( g(x,u,t) * dt ) + h( x(T) , T )
    """

    def __init__(self):

        self.EPS = 0.1

        # Target state
        self.x_target = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

        # Quadratic cost weights
        self.Q = np.diag([1.0, 1.0, 6.0, 0.1, 0.1, 0.1])
        self.R = np.diag([0.001, 0.001])

        # Optional zone of zero cost if ||dx|| < EPS
        self.ontarget_check = False

    def g(self, x, u, t=0.0, params=None):
        """Quadratic additive running cost"""

        dx = x - self.x_target

        dJ = dx.T @ self.Q @ dx + u.T @ self.R @ u

        if self.ontarget_check:
            if np.linalg.norm(dx) < self.EPS:
                dJ = 0.0

        return dJ

    def h(self, x, t=0.0, params=None):
        """Terminal cost function with zero value"""

        return 0.0


cost = CustomCostFunction()

# Gym environment

[Gymnasium](https://gymnasium.farama.org) is the standard Python API for reinforcement-learning environments (the maintained successor of OpenAI Gym). Almost every common RL library — including Stable-Baselines3, which we use below — trains against this interface: `reset`, `step`, an observation space, and an action space.

`Sys2Gym` wraps the plant and the cost as one of those environments: each step integrates $\dot x = f(x,u)$ over $\Delta t$ and returns the reward $r = -g\,\Delta t$. The remaining design choice is the **distribution of initial states** used during exploration — this is what the policy must learn to recover from.


In [ ]:
plant.x0 = np.zeros(6)  # nominal initial state: hover at the origin

env = Sys2Gym(plant, cost, dt=0.05)  # note the time step used for discrete time

# Gaussian distribution of initial states around x0
env.reset_mode = "gaussian"
env.x0_std = np.array([5.0, 5.0, 1.0, 1.0, 1.0, 0.2])

# PPO controller

We create an untrained neural policy $\pi_\theta(x)$ with PPO, then wrap it as a minilink state-feedback controller $u=\pi_\theta(x)$. Training below updates $\theta$ in place; the same `ppo_ctl` is what we plot and close the loop with.


In [ ]:
from stable_baselines3 import PPO

nn = PPO("MlpPolicy", env, verbose=1)
ppo_ctl = SB3Controller(nn, sys=plant)


# Looking at the policy

Before training, $\pi_\theta$ is essentially random. ``plot_control_law`` draws a **slice** of $u=\pi_\theta(x)$: thrust vs.\ $(\theta,\omega)$, with the other states pinned at the hover target. Re-run the same plots after training to see how the law changes.


In [ ]:
ppo_ctl.plot_control_law(x_axis=2, y_axis=5, u_axis=0)  # T1 vs (theta, omega)
ppo_ctl.plot_control_law(x_axis=2, y_axis=5, u_axis=1)  # T2 vs (theta, omega)


# Training

PPO updates a neural policy $\pi_\theta(x)$ from sampled trajectories to increase expected return (decrease $J$). Start with a shorter run, look at the policy and closed-loop behaviour below, then come back and train for more steps as needed.

In [ ]:
training_timesteps = 20000
nn.learn(training_timesteps)


In [ ]:
ppo_ctl.plot_control_law(x_axis=2, y_axis=5, u_axis=0)  # T1 vs (theta, omega)
ppo_ctl.plot_control_law(x_axis=2, y_axis=5, u_axis=1)  #

Optional extra training. Uncomment and re-run if the policy has not settled yet.

In [ ]:
# nn.learn(200000)

# Looking at the policy

Same slice after training. A stabilizing pitch loop should look like $T_1-T_2$ opposing $\theta$ and $\omega$.


In [ ]:
ppo_ctl.plot_control_law(x_axis=2, y_axis=5, u_axis=0)  # T1 vs (theta, omega)
ppo_ctl.plot_control_law(x_axis=2, y_axis=5, u_axis=1)  # T2 vs (theta, omega)


# Testing the closed-loop system

``ppo_ctl @ plant`` builds the closed-loop diagram $u=\pi_\theta(x)$. We integrate $\dot x = f(x,\pi_\theta(x))$ from an offset initial state — try other $x_0$ here.

In [ ]:
plant.x0 = np.array([-1.0, -2.0, 1.0, 0.0, 0.0, 0.0])  # initial state

cl_sys = ppo_ctl @ plant
cl_sys.name = "Drone with PPO controller"
cl_sys.plot_diagram()

In [ ]:
traj = cl_sys.compute_trajectory(tf=10.0, n_steps=1001, solver="euler")
cl_sys.plot_trajectory(traj)

**Performance**

The realized cost along the closed-loop trajectory. $\dot J = g(x,u,t)$ is the running cost at each instant; $J(t)=\int_0^t g\,d\tau$ is the cumulative performance of $\pi_\theta$ for the cost you chose.

In [ ]:
# Rebuild the applied inputs from the policy, then evaluate the cost
u_sim, _ = nn.predict(traj.x.T.astype(np.float32), deterministic=True)

plant_traj = Trajectory(t=traj.t, x=traj.x, u=u_sim.T)
plant_traj = cost.evaluate_trajectory(plant_traj)

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
axes[0].plot(plant_traj.t, plant_traj.signals["cost_rate"][0])
axes[0].set_ylabel("$\\dot{J} = g(x,u,t)$")
axes[0].grid(True, alpha=0.3)
axes[1].plot(plant_traj.t, plant_traj.signals["cost"][0])
axes[1].set_ylabel("$J = \\int g \\, dt$")
axes[1].set_xlabel("t [s]")
axes[1].grid(True, alpha=0.3)
plt.show()

print("Total trajectory cost J =", round(float(plant_traj.signals["cost"][0, -1]), 1))

**Animation of the simulation**

Replay the same closed-loop trajectory on the drone geometry.

In [ ]:
cl_sys.animate(traj)